# Notebook 05 — Fine-Tune BERT-base for AI Text Detection

**Project:** MSc AI Dissertation — AI-Generated Text Detection  
**Student:** Abdul Hannaan Mohammed | B00409227 | UWS  
**Week:** 4–5 of 13  

**Goal:** Fine-tune `bert-base-uncased` on HC3 as a second classifier for multi-model comparison.  
BERT is the standard baseline transformer — comparing it against RoBERTa shows the impact of RoBERTa's improvements.

**Reference:** Devlin et al. (2019) BERT. doi:10.18653/v1/N19-1423

## 1. Imports, Paths and Seeds

In [1]:
import os, json, random, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib; matplotlib.rcParams['figure.dpi'] = 150
import seaborn as sns
sns.set_theme(style='whitegrid')

import torch
from torch.utils.data import Dataset
from transformers import (
    BertTokenizerFast, BertForSequenceClassification,
    TrainingArguments, Trainer, EarlyStoppingCallback, set_seed
)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import wandb

# ── Paths ──────────────────────────────────────────────────────────────────────
NOTEBOOK_DIR    = os.path.dirname(os.path.abspath('__file__'))
PROJECT_ROOT    = os.path.dirname(NOTEBOOK_DIR)
DATA_PROCESSED  = os.path.join(PROJECT_ROOT, 'data', 'processed')
CHECKPOINTS_DIR = os.path.join(PROJECT_ROOT, 'models', 'checkpoints')
RESULTS_FIGS    = os.path.join(PROJECT_ROOT, 'results', 'figures')
RESULTS_METRICS = os.path.join(PROJECT_ROOT, 'results', 'metrics')
LOGS_DIR        = os.path.join(PROJECT_ROOT, 'logs')

for path in [CHECKPOINTS_DIR, RESULTS_FIGS, RESULTS_METRICS, LOGS_DIR]:
    os.makedirs(path, exist_ok=True)

# ── Reproducibility seed ────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
set_seed(SEED)

# ── Device ──────────────────────────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

Using device: cuda
GPU: NVIDIA GeForce RTX 3060 Laptop GPU
VRAM: 6.4 GB


## 2. Hyperparameters

Using identical hyperparameters to Notebook 03 (RoBERTa) so results are directly comparable.

In [2]:
MODEL_NAME   = 'bert-base-uncased'
NUM_LABELS   = 2
MAX_LENGTH   = 512
BATCH_SIZE   = 8
GRAD_ACCUM   = 4           # effective batch = 32
NUM_EPOCHS   = 3
LEARNING_RATE = 2e-5
WEIGHT_DECAY  = 0.01
WARMUP_RATIO  = 0.1
FP16          = torch.cuda.is_available()

print('=== BERT Hyperparameters ===')
print(f'  Model            : {MODEL_NAME}')
print(f'  Max length       : {MAX_LENGTH}')
print(f'  Batch size       : {BATCH_SIZE} (effective: {BATCH_SIZE*GRAD_ACCUM})')
print(f'  Epochs           : {NUM_EPOCHS}')
print(f'  Learning rate    : {LEARNING_RATE}')
print(f'  FP16             : {FP16}')
print(f'  Seed             : {SEED}')

=== BERT Hyperparameters ===
  Model            : bert-base-uncased
  Max length       : 512
  Batch size       : 8 (effective: 32)
  Epochs           : 3
  Learning rate    : 2e-05
  FP16             : True
  Seed             : 42


## 3. wandb Setup

In [3]:
wandb.login()
WANDB_PROJECT  = 'msc-ai-detection'
WANDB_RUN_NAME = 'bert-base-hc3'
print(f'wandb project: {WANDB_PROJECT} | run: {WANDB_RUN_NAME}')

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\Abdul\_netrc.
wandb: Currently logged in as: ahm11129 (ahm11129-university-of-the-west-of-scotland) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb project: msc-ai-detection | run: bert-base-hc3


## 4. Load Data

In [4]:
train_df = pd.read_csv(os.path.join(DATA_PROCESSED, 'train.csv'))
val_df   = pd.read_csv(os.path.join(DATA_PROCESSED, 'val.csv'))

print(f'Train : {len(train_df):,}  | Human: {(train_df["label"]==0).sum():,}  AI: {(train_df["label"]==1).sum():,}')
print(f'Val   : {len(val_df):,}   | Human: {(val_df["label"]==0).sum():,}   AI: {(val_df["label"]==1).sum():,}')

Train : 55,156  | Human: 36,808  AI: 18,348
Val   : 11,819   | Human: 7,888   AI: 3,931


## 5. Tokenise and Build Datasets

In [5]:
class TextDataset(Dataset):
    """PyTorch Dataset wrapping tokenised text and labels."""
    def __init__(self, df, tokeniser, max_length):
        self.labels    = df['label'].tolist()
        self.encodings = tokeniser(
            df['text'].tolist(),
            max_length=max_length,
            padding='max_length',
            truncation=True
        )

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'input_ids':      torch.tensor(self.encodings['input_ids'][idx],      dtype=torch.long),
            'attention_mask': torch.tensor(self.encodings['attention_mask'][idx], dtype=torch.long),
            'token_type_ids': torch.tensor(self.encodings['token_type_ids'][idx], dtype=torch.long),
            'labels':         torch.tensor(self.labels[idx],                      dtype=torch.long)
        }


print('Loading BERT tokeniser...')
tokeniser = BertTokenizerFast.from_pretrained(MODEL_NAME)

print('Building datasets (tokenising — ~30 seconds)...')
train_dataset = TextDataset(train_df, tokeniser, MAX_LENGTH)
val_dataset   = TextDataset(val_df,   tokeniser, MAX_LENGTH)

print(f'Train dataset: {len(train_dataset):,} samples')
print(f'Val dataset  : {len(val_dataset):,} samples')

Loading BERT tokeniser...


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

Building datasets (tokenising — ~30 seconds)...
Train dataset: 55,156 samples
Val dataset  : 11,819 samples


## 6. Load BERT Model

In [6]:
print('Loading bert-base-uncased (~440MB download on first run)...')
model = BertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=NUM_LABELS)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model loaded. Trainable parameters: {n_params:,}  ({n_params/1e6:.1f}M)')

Loading bert-base-uncased (~440MB download on first run)...


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model loaded. Trainable parameters: 109,483,778  (109.5M)


## 7. Compute Metrics Function

In [7]:
def compute_metrics(eval_pred):
    """Compute accuracy, precision, recall, F1 after each eval epoch."""
    logits, labels = eval_pred
    predictions    = np.argmax(logits, axis=-1)
    accuracy       = accuracy_score(labels, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average='binary', pos_label=1
    )
    return {
        'accuracy' : round(accuracy,  4),
        'precision': round(precision, 4),
        'recall'   : round(recall,    4),
        'f1'       : round(f1,        4)
    }

print('compute_metrics defined.')

compute_metrics defined.


## 8. Train

**Expected time:** ~1.5–2 hours on RTX 3060 6GB  

> SCREENSHOT REMINDER: While training, go to wandb.ai and screenshot the dashboard.  
> Save as: `results/screenshots/06_bert_training_curves.png`

In [10]:
CHECKPOINT_DIR = os.path.join(CHECKPOINTS_DIR, 'bert-hc3')

training_args = TrainingArguments(
    output_dir=CHECKPOINT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,
    fp16=FP16,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    greater_is_better=True,
    logging_dir=LOGS_DIR,
    logging_steps=50,
    report_to='wandb',
    run_name=WANDB_RUN_NAME,
    seed=SEED,
    dataloader_num_workers=0,
    save_total_limit=2,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

print('Starting BERT training...')
train_result = trainer.train()

print('\n=== Training Complete ===')
print(f'Total time  : {train_result.metrics["train_runtime"]:.0f} seconds')
print(f'Samples/sec : {train_result.metrics["train_samples_per_second"]:.2f}')

Starting BERT training...


  0%|          | 0/5169 [00:00<?, ?it/s]

KeyboardInterrupt: 

## 9. Save Best Model

In [ ]:
BEST_MODEL_DIR = os.path.join(CHECKPOINTS_DIR, 'bert-hc3-best')
trainer.save_model(BEST_MODEL_DIR)
tokeniser.save_pretrained(BEST_MODEL_DIR)
print(f'Best model saved: {BEST_MODEL_DIR}')

## 10. Save Metrics and Plot Training Curves

> SCREENSHOT: Screenshot the training curves plot.  
> Save as: `results/screenshots/06_bert_training_curves.png`

In [ ]:
history    = trainer.state.log_history
eval_logs  = [h for h in history if 'eval_loss' in h]
train_logs = [h for h in history if 'loss' in h and 'eval_loss' not in h]

epochs_data = [{
    'epoch'         : ev.get('epoch', i+1),
    'eval_loss'     : ev.get('eval_loss'),
    'eval_accuracy' : ev.get('eval_accuracy'),
    'eval_f1'       : ev.get('eval_f1'),
    'eval_precision': ev.get('eval_precision'),
    'eval_recall'   : ev.get('eval_recall'),
} for i, ev in enumerate(eval_logs)]

history_df = pd.DataFrame(epochs_data)
history_df.to_csv(os.path.join(RESULTS_METRICS, 'bert_training_history.csv'), index=False)

# Plot
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('BERT-base Fine-Tuning — Training Curves', fontsize=13, fontweight='bold')

epochs = history_df['epoch'].tolist()
axes[0].plot(epochs, history_df['eval_loss'], marker='o', color='#DD8452', linewidth=2)
axes[0].set_title('Validation Loss'); axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')

axes[1].plot(epochs, history_df['eval_accuracy'], marker='o', color='#4C72B0', linewidth=2, label='Accuracy')
axes[1].plot(epochs, history_df['eval_f1'],       marker='s', color='#55A868', linewidth=2, label='F1')
axes[1].set_title('Validation Accuracy & F1'); axes[1].set_xlabel('Epoch')
axes[1].set_ylim(0, 1.05); axes[1].legend()

plt.tight_layout()
fig_path = os.path.join(RESULTS_FIGS, 'fig_bert_training_curves.png')
plt.savefig(fig_path, bbox_inches='tight', dpi=150)
print(f'SCREENSHOT NOW: results/screenshots/06_bert_training_curves.png for Chapter 4')
plt.show()

# Best epoch metrics
best = history_df.loc[history_df['eval_f1'].idxmax()]
bert_metrics = {
    'model'      : 'bert-base-uncased',
    'dataset'    : 'HC3',
    'best_epoch' : int(best['epoch']),
    'val_accuracy' : float(best['eval_accuracy']),
    'val_precision': float(best['eval_precision']),
    'val_recall'   : float(best['eval_recall']),
    'val_f1'       : float(best['eval_f1']),
    'best_model_dir': BEST_MODEL_DIR
}
with open(os.path.join(RESULTS_METRICS, 'bert_final_metrics.json'), 'w') as f:
    json.dump(bert_metrics, f, indent=2)

print('\n=== BERT BEST VALIDATION METRICS ===')
print(f'  Accuracy  : {bert_metrics["val_accuracy"]:.4f}')
print(f'  Precision : {bert_metrics["val_precision"]:.4f}')
print(f'  Recall    : {bert_metrics["val_recall"]:.4f}')
print(f'  F1        : {bert_metrics["val_f1"]:.4f}')

wandb.finish()

## Notebook Summary

### Files produced
| File | Location |
|------|----------|
| `bert-hc3-best/` | `models/checkpoints/` |
| `bert_training_history.csv` | `results/metrics/` |
| `bert_final_metrics.json` | `results/metrics/` |
| `fig_bert_training_curves.png` | `results/figures/` |

### Next step
**Notebook 06:** `06_train_distilbert.ipynb` — Fine-tune DistilBERT as the lightweight model.